In [24]:
# ============================================================
# D5 — Branch C: Structural Markdown conversion
# 0. Imports and frozen experimental configuration
# ============================================================
!pip -q install pymupdf

import json
import hashlib
import platform
import re
import sys
import unicodedata

from collections import Counter
from datetime import datetime
from pathlib import Path

import fitz
import pandas as pd

from google.colab import files

DOCUMENT_ID = "D5"
DOCUMENT_NAME = "IMF Country Focus — Booming India at risk of overheating"

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

EXPECTED_SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = "ce59b2fb51cfcee5b579dd713c59ae5a5e14b57f66eab40d4cbaee151a62064d"
EXPECTED_PAGE_COUNT = 2

EXPECTED_RECORD_COUNT = 44

EXPECTED_CATEGORY_COUNTS = {
    "Main policy measure": 4,
    "Country profile": 6,
    "Narrative quantitative observation": 16,
    "Statistical table observation": 18
}

EXPECTED_FIELDS = [
    "Category",
    "Indicator or Policy Area",
    "Value",
    "Unit",
    "Qualifier",
    "Reference Period",
    "Description",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

KNOWN_HEADINGS = [
    "Booming India at risk of overheating",
    "India at a glance",
    "Taking off",
    "Ease up on the monetary accelerator",
    "Reduce debt to finance development",
    "Develop broader and deeper capital markets",
    "Promote job growth and bolster the infrastructure",
    "Inflation risks"
]

EXPECTED_COMPONENTS = {
    "article_title": "Booming India at risk of overheating",
    "country_profile": "India at a glance",
    "chart": "Taking off",
    "monetary_section": "Ease up on the monetary accelerator",
    "fiscal_section": "Reduce debt to finance development",
    "capital_markets_section": "Develop broader and deeper capital markets",
    "employment_section": "Promote job growth and bolster the infrastructure",
    "statistical_table": "Inflation risks",
    "author": "Charles Kramer",
    "publication_date": "April 11, 2007"
}

OUTPUT_DIR = Path("outputs_D5_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configured:", DOCUMENT_ID, BRANCH)
print("Parent branch:", PARENT_BRANCH)
print("Expected source pages:", EXPECTED_PAGE_COUNT)


Configured: D5 C
Parent branch: B
Expected source pages: 2


In [25]:
# ============================================================
# 1. Upload original D5 PDF and required Branch B artefacts
# ============================================================
# Upload exactly:
#   1) original D5 PDF
#   2) D5_branch_B_structural_markdown.md
#   3) D5_branch_B_conversion_integrity.json

uploaded = files.upload()
names = list(uploaded.keys())

pdf_files = [Path(f) for f in names if f.lower().endswith(".pdf")]
md_files = [Path(f) for f in names if f.lower().endswith(".md")]
json_files = [Path(f) for f in names if f.lower().endswith(".json")]

if len(pdf_files) != 1 or len(md_files) != 1 or len(json_files) != 1:
    raise ValueError(
        "Upload exactly one PDF, one Branch B structural Markdown file, "
        "and one Branch B conversion-integrity JSON file."
    )

SOURCE_PATH = pdf_files[0]
BRANCH_B_REPRESENTATION_PATH = md_files[0]
BRANCH_B_CHECK_PATH = json_files[0]

print("Source:", SOURCE_PATH.name)
print("Branch B representation:", BRANCH_B_REPRESENTATION_PATH.name)
print("Branch B integrity:", BRANCH_B_CHECK_PATH.name)


Saving D5_branch_B_conversion_integrity.json to D5_branch_B_conversion_integrity (1).json
Saving D5_branch_B_structural_markdown.md to D5_branch_B_structural_markdown (1).md
Saving D5 - IMF “Country Focus” – India Article.pdf to D5 - IMF “Country Focus” – India Article (1).pdf
Source: D5 - IMF “Country Focus” – India Article (1).pdf
Branch B representation: D5_branch_B_structural_markdown (1).md
Branch B integrity: D5_branch_B_conversion_integrity (1).json


In [26]:
# ============================================================
# 2. Verify frozen source identity and Branch B parent integrity
# ============================================================
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

if SOURCE_PATH.suffix.lower() != EXPECTED_SOURCE_FORMAT:
    raise ValueError("Unexpected D5 source format.")

SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if not SOURCE_HASH_MATCH:
    raise ValueError("Uploaded PDF does not match the frozen D5 source identity.")

with open(BRANCH_B_CHECK_PATH, "r", encoding="utf-8") as f:
    branch_b_check = json.load(f)

if branch_b_check.get("document_id") != DOCUMENT_ID:
    raise ValueError("Branch B integrity artefact belongs to another document.")

if branch_b_check.get("branch") != "B":
    raise ValueError("Uploaded integrity artefact is not from Branch B.")

if branch_b_check.get("source_sha256") != SOURCE_SHA256:
    raise ValueError("Branch B was generated from a different D5 source identity.")

if not branch_b_check.get("conversion_integrity_passed", False):
    raise ValueError("Branch B parent representation did not pass conversion integrity.")

SOURCE_B_MARKDOWN = BRANCH_B_REPRESENTATION_PATH.read_text(encoding="utf-8")

if not SOURCE_B_MARKDOWN.strip():
    raise ValueError("Uploaded Branch B structural Markdown is empty.")

SOURCE_B_SHA256 = sha256_text(SOURCE_B_MARKDOWN)

print("Frozen source identity verified.")
print("Branch B conversion integrity verified.")
print("Uploaded Branch B representation SHA-256:", SOURCE_B_SHA256)


Frozen source identity verified.
Branch B conversion integrity verified.
Uploaded Branch B representation SHA-256: 1e226dfa7e39c992a1ab2a90334ac89647a9055a6220c05d0dd2d3b31e963991


In [27]:
# ============================================================
# 3. Reproduce the exact Branch B structural state
# ============================================================
document = fitz.open(SOURCE_PATH)

if len(document) != EXPECTED_PAGE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages; observed {len(document)}."
    )

if document.needs_pass:
    raise ValueError("The D5 source requires a password.")

page_texts = [
    page.get_text("text", sort=True)
    for page in document
]

full_text = "\n".join(page_texts)

component_checks = {
    key: marker in full_text
    for key, marker in EXPECTED_COMPONENTS.items()
}

if not bool(full_text.strip()):
    raise ValueError("D5 should contain a machine-readable text layer.")

if not all(component_checks.values()):
    raise ValueError("One or more expected D5 source components are missing.")


def preserve_block_text(value):
    if value is None:
        return None

    text = str(value).replace("\r\n", "\n").replace("\r", "\n")
    lines = [line.rstrip() for line in text.splitlines()]
    text = "\n".join(lines).strip()

    return text if text else None


block_records = []

for page_index, page in enumerate(document):
    page_number = page_index + 1

    for block_position, block in enumerate(
        page.get_text("blocks", sort=False)
    ):
        text = preserve_block_text(block[4])

        if not text:
            continue

        block_records.append({
            "Page Number": page_number,
            "Block Position": block_position,
            "Block Number": int(block[5]) if len(block) > 5 else block_position,
            "Block Type": int(block[6]) if len(block) > 6 else 0,
            "X0": float(block[0]),
            "Y0": float(block[1]),
            "X1": float(block[2]),
            "Y1": float(block[3]),
            "Text": text
        })

blocks_df = pd.DataFrame(block_records)


def layout_region(page_number, x0, y0, page_width, page_height):
    rx = x0 / page_width
    ry = y0 / page_height

    if ry < 0.08:
        return "Page header"

    if rx < 0.50:
        return f"Page {page_number} left region"

    return f"Page {page_number} right region"


ordered_parts = []

for page_number in range(1, EXPECTED_PAGE_COUNT + 1):
    page = document[page_number - 1]

    subset = blocks_df[
        blocks_df["Page Number"] == page_number
    ].copy()

    subset["Layout Region"] = subset.apply(
        lambda row: layout_region(
            page_number,
            row["X0"],
            row["Y0"],
            page.rect.width,
            page.rect.height
        ),
        axis=1
    )

    region_rank = {
        "Page header": 0,
        f"Page {page_number} left region": 1,
        f"Page {page_number} right region": 2
    }

    subset["_region_rank"] = subset["Layout Region"].map(
        region_rank
    ).fillna(9)

    subset = subset.sort_values(
        ["_region_rank", "Y0", "X0", "Block Position"]
    )

    ordered_parts.append(subset)

ordered_blocks_df = pd.concat(
    ordered_parts,
    ignore_index=True
)

ordered_blocks_df["Structural Order"] = range(
    1,
    len(ordered_blocks_df) + 1
)


# Reproduce Branch B Inflation risks table reconstruction.
page2 = document[1]
page2_dict = page2.get_text("dict")

spans = []

for block in page2_dict["blocks"]:
    for line in block.get("lines", []):
        for span in line.get("spans", []):
            text = span.get("text", "")
            if text.strip():
                spans.append({
                    "text": text.strip(),
                    "size": float(span["size"]),
                    "x0": float(span["bbox"][0]),
                    "y0": float(span["bbox"][1]),
                    "x1": float(span["bbox"][2]),
                    "y1": float(span["bbox"][3])
                })

span_df = pd.DataFrame(spans)

table_spans = span_df[
    (span_df["x0"] < 310)
    & (span_df["y0"] >= 600)
    & (span_df["y0"] <= 725)
].copy()

period_spans = table_spans[
    table_spans["text"].str.fullmatch(
        r"(?:19|20)\d{2}/\d{2}",
        na=False
    )
].sort_values("x0")

if len(period_spans) != 3:
    raise ValueError("Could not reproduce the three D5 table periods.")

period_headers = period_spans["text"].tolist()
period_x = period_spans["x0"].tolist()

status_spans = table_spans[
    table_spans["text"].isin(["Prov.", "Est."])
].copy()

column_status = {
    period: None
    for period in period_headers
}

for _, status in status_spans.iterrows():
    nearest = min(
        range(len(period_x)),
        key=lambda i: abs(period_x[i] - status["x0"])
    )
    column_status[period_headers[nearest]] = status["text"]

display_period_headers = [
    period
    if column_status[period] is None
    else f"{period} {column_status[period]}"
    for period in period_headers
]

unit_markers = []

for _, group in table_spans.groupby(
    table_spans["y0"].round(0)
):
    joined = " ".join(
        group.sort_values("x0")["text"].tolist()
    ).strip()

    if joined in {
        "(percent change)",
        "(percent of GDP)",
        "(million dollars)"
    }:
        unit_markers.append({
            "y": float(group["y0"].mean()),
            "unit": joined.strip("()")
        })

unit_markers = sorted(
    unit_markers,
    key=lambda x: x["y"]
)

data_rows = []

for _, group in table_spans.groupby(
    table_spans["y0"].round(0)
):
    group = group.sort_values("x0")
    normal_spans = group[group["size"] >= 6.0]

    numeric_candidates = normal_spans[
        normal_spans["text"].str.fullmatch(
            r"[–−-]?\d+(?:\.\d+)?",
            na=False
        )
        & (normal_spans["x0"] >= 170)
        & (normal_spans["x0"] <= 295)
    ]

    if len(numeric_candidates) != 3:
        continue

    values = numeric_candidates["text"].tolist()

    label_spans = normal_spans[
        normal_spans["x0"] < 170
    ]

    label = " ".join(
        label_spans["text"].tolist()
    ).strip()

    if not label:
        continue

    row_y = float(group["y0"].mean())

    inline_unit_match = re.search(
        r"\(([^()]*(?:dollars|percent)[^()]*)\)",
        label,
        flags=re.IGNORECASE
    )

    if inline_unit_match:
        unit = inline_unit_match.group(1)
        label = re.sub(
            r"\s*\([^()]*\)\s*$",
            "",
            label
        ).strip()
    else:
        prior_markers = [
            marker
            for marker in unit_markers
            if marker["y"] < row_y
        ]
        unit = (
            prior_markers[-1]["unit"]
            if prior_markers
            else None
        )

    data_rows.append({
        "Indicator": label,
        "Unit": unit,
        display_period_headers[0]: values[0],
        display_period_headers[1]: values[1],
        display_period_headers[2]: values[2]
    })

table_df = pd.DataFrame(data_rows)

EXPECTED_TABLE_INDICATORS = [
    "Real GDP",
    "Wholesale prices",
    "General government debt",
    "Current account balance",
    "External debt",
    "Gross reserves"
]

if table_df["Indicator"].tolist() != EXPECTED_TABLE_INDICATORS:
    raise ValueError("Could not reproduce the Branch B table rows in order.")

footnote_spans = span_df[
    (span_df["x0"] < 310)
    & (span_df["y0"] >= 710)
    & (span_df["y0"] <= 725)
].sort_values(["y0", "x0"])

footnote_text = " ".join(
    footnote_spans["text"].tolist()
).strip()


def first_line_heading(text):
    lines = text.splitlines()

    if not lines:
        return None, text

    first = lines[0].strip()

    for heading in KNOWN_HEADINGS:
        if first.casefold() == heading.casefold():
            remainder = "\n".join(
                lines[1:]
            ).strip()

            return heading, remainder

    return None, text


def escape_md_cell(value):
    if value is None:
        return ""

    return (
        str(value)
        .replace("|", "\\|")
        .replace("\n", "<br>")
    )


table_headers = table_df.columns.tolist()

table_md = [
    "| " + " | ".join(table_headers) + " |",
    "| " + " | ".join(["---"] * len(table_headers)) + " |"
]

for _, row in table_df.iterrows():
    table_md.append(
        "| "
        + " | ".join(
            escape_md_cell(row[col])
            for col in table_headers
        )
        + " |"
    )

markdown_lines = [
    "# D5 — Booming India at risk of overheating",
    "",
    "> Complete structural conversion of the original two-page PDF.",
    "> All non-empty source text blocks are retained.",
    ""
]

for page_number in range(1, EXPECTED_PAGE_COUNT + 1):
    markdown_lines += [
        f"## Source Page {page_number}",
        ""
    ]

    page_subset = ordered_blocks_df[
        ordered_blocks_df["Page Number"] == page_number
    ]

    for _, row in page_subset.iterrows():
        markdown_lines += [
            f"### Source Block {int(row['Structural Order'])}",
            f"- Layout Region: `{row['Layout Region']}`",
            (
                f"- Bounding Box: "
                f"`({row['X0']:.2f}, {row['Y0']:.2f}, "
                f"{row['X1']:.2f}, {row['Y1']:.2f})`"
            ),
            ""
        ]

        heading, remainder = first_line_heading(
            row["Text"]
        )

        if heading is not None:
            markdown_lines += [
                f"#### {heading}",
                ""
            ]

            if remainder:
                markdown_lines += [
                    "```text",
                    remainder,
                    "```",
                    ""
                ]

        else:
            markdown_lines += [
                "```text",
                row["Text"],
                "```",
                ""
            ]

    if page_number == 2:
        markdown_lines += [
            "## Structural interpretation — Inflation risks table",
            "",
            *table_md,
            "",
            "### Table footnote",
            "",
            "```text",
            footnote_text,
            "```",
            ""
        ]

REPRODUCED_BRANCH_B_MARKDOWN = (
    "\n".join(markdown_lines).rstrip()
    + "\n"
)

REPRODUCED_B_SHA256 = sha256_text(
    REPRODUCED_BRANCH_B_MARKDOWN
)

print("Reproduced Branch B records/blocks:", len(ordered_blocks_df))
print("Reproduced Branch B SHA-256:", REPRODUCED_B_SHA256)


Reproduced Branch B records/blocks: 53
Reproduced Branch B SHA-256: 1e226dfa7e39c992a1ab2a90334ac89647a9055a6220c05d0dd2d3b31e963991


In [28]:
# ============================================================
# 4. Verify exact Branch B parent equivalence
# ============================================================
PARENT_EQUIVALENCE_PASSED = (
    REPRODUCED_BRANCH_B_MARKDOWN
    == SOURCE_B_MARKDOWN
)

parent_check = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "parent_branch": PARENT_BRANCH,

    "source_sha256": SOURCE_SHA256,

    "branch_B_conversion_integrity_passed":
        bool(
            branch_b_check.get(
                "conversion_integrity_passed",
                False
            )
        ),

    "uploaded_branch_B_sha256":
        SOURCE_B_SHA256,

    "reproduced_branch_B_sha256":
        REPRODUCED_B_SHA256,

    "branch_B_representation_exactly_reproduced":
        PARENT_EQUIVALENCE_PASSED,

    "source_page_count":
        len(document),

    "source_block_count":
        int(len(ordered_blocks_df)),

    "table_indicator_count":
        int(len(table_df)),

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED
}

PARENT_CHECK_PATH = (
    OUTPUT_DIR
    / "D5_branch_C_parent_B_equivalence_check.json"
)

PARENT_CHECK_PATH.write_text(
    json.dumps(
        parent_check,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        parent_check,
        indent=2,
        ensure_ascii=False
    )
)

if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError(
        "The uploaded Branch B representation does not exactly match "
        "the representation reproduced from the frozen D5 PDF."
    )


{
  "document_id": "D5",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "ce59b2fb51cfcee5b579dd713c59ae5a5e14b57f66eab40d4cbaee151a62064d",
  "branch_B_conversion_integrity_passed": true,
  "uploaded_branch_B_sha256": "1e226dfa7e39c992a1ab2a90334ac89647a9055a6220c05d0dd2d3b31e963991",
  "reproduced_branch_B_sha256": "1e226dfa7e39c992a1ab2a90334ac89647a9055a6220c05d0dd2d3b31e963991",
  "branch_B_representation_exactly_reproduced": true,
  "source_page_count": 2,
  "source_block_count": 53,
  "table_indicator_count": 6,
  "parent_equivalence_passed": true
}


In [29]:
# ============================================================
# 5. Define deterministic D5 Branch C normalisation
# ============================================================
#
# D5 uses a deliberately conservative normalisation layer.
# It preserves the complete Branch B structural scaffold and source
# exposure while reducing representational variation inside text.
#
# No line-break hyphenation repair or paragraph merging is applied,
# because those operations can change source wording and were not
# needed to define Branch C consistently after D1–D4.
# ============================================================

UNICODE_SPACE_CHARACTERS = [
    "\u00a0",
    "\u1680",
    "\u2000",
    "\u2001",
    "\u2002",
    "\u2003",
    "\u2004",
    "\u2005",
    "\u2006",
    "\u2007",
    "\u2008",
    "\u2009",
    "\u200a",
    "\u202f",
    "\u205f",
    "\u3000"
]

APOSTROPHE_REPLACEMENTS = {
    "’": "'",
    "‘": "'",
    "‛": "'",
    "´": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "‐": "-",
    "‑": "-",
    "‒": "-",
    "–": "-",
    "—": "-",
    "−": "-"
}


def normalise_text_representation(text):
    """
    Deterministic representation-level normalisation.

    Allowed:
    - Unicode NFKC
    - Unicode-space standardisation
    - typographic apostrophe standardisation
    - dash/minus-glyph standardisation
    - soft-hyphen removal
    - line-ending standardisation
    - horizontal whitespace collapse
    - removal of trailing whitespace

    Not allowed:
    - semantic rewriting
    - paragraph merging
    - line-break hyphenation repair
    - value calculation/conversion
    - chart-value estimation
    """

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(
            character,
            " "
        )

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    # Soft hyphen is a discretionary rendering character.
    text = text.replace(
        "\u00ad",
        ""
    )

    text = (
        text
        .replace("\r\n", "\n")
        .replace("\r", "\n")
    )

    lines = []

    for line in text.splitlines():
        line = re.sub(
            r"[ \t\f\v]+",
            " ",
            line
        ).rstrip()

        lines.append(
            line
        )

    return "\n".join(lines)


def normalise_complete_representation(text):
    """
    Apply the declared deterministic Branch C normalisation to the
    complete Branch B representation without filtering content.
    """

    text = normalise_text_representation(
        text
    )

    # Preserve Markdown boundaries but standardise excessive blank lines.
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip() + "\n"


In [30]:
# ============================================================
# 6. Apply Branch C deterministic normalisation
# ============================================================
NORMALISED_MARKDOWN = (
    normalise_complete_representation(
        SOURCE_B_MARKDOWN
    )
)

if not NORMALISED_MARKDOWN.strip():
    raise ValueError(
        "D5 Branch C normalisation produced an empty representation."
    )

print(
    "Branch B characters:",
    len(SOURCE_B_MARKDOWN)
)

print(
    "Branch C characters:",
    len(NORMALISED_MARKDOWN)
)

print(
    "Branch B source-block markers:",
    len(
        re.findall(
            r"^### Source Block \d+$",
            SOURCE_B_MARKDOWN,
            flags=re.MULTILINE
        )
    )
)

print(
    "Branch C source-block markers:",
    len(
        re.findall(
            r"^### Source Block \d+$",
            NORMALISED_MARKDOWN,
            flags=re.MULTILINE
        )
    )
)


Branch B characters: 15700
Branch C characters: 15692
Branch B source-block markers: 53
Branch C source-block markers: 53


In [31]:
# ============================================================
# 7. Verify Branch C normalisation integrity
# ============================================================
#
# Transformation-aware integrity principle:
#
# Branch C is allowed to change Unicode punctuation, soft hyphens
# and spacing. Therefore raw-string equality is NOT required.
#
# Instead, we verify:
#   1. exact Branch B parent lineage;
#   2. page and source-block sequence preservation;
#   3. deterministic equivalence to the declared Branch C function;
#   4. preservation of required named source components;
#   5. preservation of the reconstructed statistical table;
#   6. preservation of value-bearing numerical information after
#      canonicalising only the transformations Branch C is allowed to make.
#
# This follows the lesson from D1–D4: do not require preservation
# of a raw property that Branch C intentionally normalises.
# ============================================================


# ------------------------------------------------------------
# A. Verify page boundaries and source-block order
# ------------------------------------------------------------

PAGE_PATTERN = re.compile(
    r"^## Source Page (\d+)$",
    flags=re.MULTILINE
)

BLOCK_PATTERN = re.compile(
    r"^### Source Block (\d+)$",
    flags=re.MULTILINE
)

parent_pages = PAGE_PATTERN.findall(
    SOURCE_B_MARKDOWN
)

branch_c_pages = PAGE_PATTERN.findall(
    NORMALISED_MARKDOWN
)

page_sequence_preserved = (
    parent_pages
    == branch_c_pages
    == [str(i) for i in range(1, EXPECTED_PAGE_COUNT + 1)]
)

parent_blocks = BLOCK_PATTERN.findall(
    SOURCE_B_MARKDOWN
)

branch_c_blocks = BLOCK_PATTERN.findall(
    NORMALISED_MARKDOWN
)

source_block_sequence_preserved = (
    parent_blocks
    == branch_c_blocks
)

source_block_count_preserved = (
    len(parent_blocks)
    == len(branch_c_blocks)
    == len(ordered_blocks_df)
)


# ------------------------------------------------------------
# B. Verify that Branch C is exactly the output of the declared
#    deterministic normalisation function
# ------------------------------------------------------------

EXPECTED_NORMALISED_MARKDOWN = (
    normalise_complete_representation(
        SOURCE_B_MARKDOWN
    )
)

deterministic_representation_verified = (
    NORMALISED_MARKDOWN
    == EXPECTED_NORMALISED_MARKDOWN
)


# ------------------------------------------------------------
# C. Verify required document components after applying the same
#    canonical normalisation to component markers
# ------------------------------------------------------------

normalised_component_checks = {}

for key, marker in EXPECTED_COMPONENTS.items():

    canonical_marker = (
        normalise_text_representation(
            marker
        )
        .strip()
        .casefold()
    )

    canonical_representation = (
        NORMALISED_MARKDOWN.casefold()
    )

    normalised_component_checks[key] = (
        canonical_marker
        in canonical_representation
    )

all_expected_components_preserved = all(
    normalised_component_checks.values()
)


# ------------------------------------------------------------
# D. Verify reconstructed Inflation risks table structure
# ------------------------------------------------------------

TABLE_HEADING = (
    "## Structural interpretation - Inflation risks table"
)

table_heading_preserved = (
    TABLE_HEADING
    in NORMALISED_MARKDOWN
)

expected_table_row_labels = [
    "Real GDP",
    "Wholesale prices",
    "General government debt",
    "Current account balance",
    "External debt",
    "Gross reserves"
]

table_row_checks = {
    label: (
        f"| {label} |"
        in NORMALISED_MARKDOWN
    )
    for label in expected_table_row_labels
}

all_table_rows_preserved = all(
    table_row_checks.values()
)

table_footnote_preserved = (
    "### Table footnote"
    in NORMALISED_MARKDOWN
)


# ------------------------------------------------------------
# E. Value-bearing numerical preservation
# ------------------------------------------------------------
#
# Avoid a generic every-integer check. It is too broad for mixed
# magazine/PDF content and can be affected by harmless Unicode
# normalisation. Instead verify D5-relevant quantitative forms.
# ------------------------------------------------------------

VALUE_PATTERNS = {
    "percent_expressions":
        r"\b\d+(?:\.\d+)?\s*percent\b",

    "basis_point_expressions":
        r"\b\d+(?:\.\d+)?\s*basis\s+points?\b",

    "currency_values":
        r"\$\s*\d[\d,]*(?:\.\d+)?",

    "million_billion_expressions":
        r"\b\d+(?:\.\d+)?\s*(?:million|billion)\b",

    "fiscal_periods":
        r"\b(?:19|20)\d{2}/\d{2}\b",

    "decimal_table_values":
        r"(?<![\w])[-–−]?\d+\.\d+(?![\w])"
}


def canonicalise_value_token(token):
    token = normalise_text_representation(
        token
    )

    token = re.sub(
        r"[ \t]+",
        " ",
        token
    )

    token = re.sub(
        r"\$\s+",
        "$",
        token
    )

    return token.strip().casefold()


numeric_preservation = {}

for label, pattern in VALUE_PATTERNS.items():

    before = [
        canonicalise_value_token(token)
        for token in re.findall(
            pattern,
            SOURCE_B_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    after = [
        canonicalise_value_token(token)
        for token in re.findall(
            pattern,
            NORMALISED_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    before_counter = Counter(
        before
    )

    after_counter = Counter(
        after
    )

    missing = list(
        (
            before_counter
            - after_counter
        ).elements()
    )

    added = list(
        (
            after_counter
            - before_counter
        ).elements()
    )

    numeric_preservation[label] = {
        "count_before":
            len(before),

        "count_after":
            len(after),

        "missing_token_count":
            len(missing),

        "added_token_count":
            len(added),

        "passed":
            len(missing) == 0
            and len(added) == 0
    }


numeric_values_preserved = all(
    item["passed"]
    for item in numeric_preservation.values()
)


# ------------------------------------------------------------
# F. Verify the reconstructed table's canonical values directly
# ------------------------------------------------------------

EXPECTED_TABLE_VALUES_CANONICAL = {
    "Real GDP":
        ["7.5", "9.0", "8.9"],

    "Wholesale prices":
        ["6.5", "4.4", "6.4"],

    "General government debt":
        ["85.7", "81.9", "79.3"],

    "Current account balance":
        ["-2.5", "-9.1", "-22.7"],

    "External debt":
        ["17.7", "15.7", "18.1"],

    "Gross reserves":
        ["141.5", "151.6", "198.6"]
}

table_value_checks = {}

for indicator, values in EXPECTED_TABLE_VALUES_CANONICAL.items():

    row = table_df[
        table_df["Indicator"] == indicator
    ]

    if len(row) != 1:
        table_value_checks[indicator] = False
        continue

    observed = [
        canonicalise_value_token(v)
        for v in row.iloc[0, 2:].tolist()
    ]

    expected = [
        canonicalise_value_token(v)
        for v in values
    ]

    table_value_checks[indicator] = (
        observed == expected
    )

all_table_values_preserved = all(
    table_value_checks.values()
)


# ------------------------------------------------------------
# G. Final integrity decision
# ------------------------------------------------------------

normalisation_integrity_passed = bool(

    PARENT_EQUIVALENCE_PASSED

    and page_sequence_preserved

    and source_block_sequence_preserved

    and source_block_count_preserved

    and deterministic_representation_verified

    and all_expected_components_preserved

    and table_heading_preserved

    and all_table_rows_preserved

    and table_footnote_preserved

    and numeric_values_preserved

    and all_table_values_preserved
)


normalisation_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "page_sequence_preserved":
        page_sequence_preserved,

    "parent_page_markers":
        parent_pages,

    "branch_C_page_markers":
        branch_c_pages,

    "source_block_count":
        int(len(parent_blocks)),

    "source_block_count_preserved":
        source_block_count_preserved,

    "source_block_sequence_preserved":
        source_block_sequence_preserved,

    "deterministic_representation_verified":
        deterministic_representation_verified,

    "normalised_component_checks":
        normalised_component_checks,

    "all_expected_components_preserved":
        all_expected_components_preserved,

    "table_heading_preserved":
        table_heading_preserved,

    "table_row_checks":
        table_row_checks,

    "all_table_rows_preserved":
        all_table_rows_preserved,

    "table_footnote_preserved":
        table_footnote_preserved,

    "numeric_token_preservation":
        numeric_preservation,

    "numeric_values_preserved":
        numeric_values_preserved,

    "table_value_checks":
        table_value_checks,

    "all_table_values_preserved":
        all_table_values_preserved,

    "complete_source_content_retained":
        True,

    "page_cropping_applied":
        False,

    "source_block_filtering_applied":
        False,

    "out_of_scope_content_retained":
        True,

    "unicode_nfkc_normalisation_applied":
        True,

    "unicode_space_standardisation_applied":
        True,

    "apostrophe_standardisation_applied":
        True,

    "dash_and_minus_standardisation_applied":
        True,

    "soft_hyphen_removal_applied":
        True,

    "line_endings_standardised":
        True,

    "horizontal_whitespace_normalisation_applied":
        True,

    "paragraph_line_merging_applied":
        False,

    "line_break_hyphenation_repair_applied":
        False,

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "chart_values_estimated":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_integrity_passed
}


NORMALISATION_CHECK_PATH = (
    OUTPUT_DIR
    / "D5_branch_C_normalisation_check.json"
)

NORMALISATION_CHECK_PATH.write_text(
    json.dumps(
        normalisation_check,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        normalisation_check,
        indent=2,
        ensure_ascii=False
    )
)

if not normalisation_integrity_passed:
    raise ValueError(
        "D5 Branch C normalisation-integrity checks failed. "
        "Inspect page/block preservation, component preservation, "
        "table preservation and transformation-aware numerical checks."
    )


{
  "document_id": "D5",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "page_sequence_preserved": true,
  "parent_page_markers": [
    "1",
    "2"
  ],
  "branch_C_page_markers": [
    "1",
    "2"
  ],
  "source_block_count": 53,
  "source_block_count_preserved": true,
  "source_block_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "normalised_component_checks": {
    "article_title": true,
    "country_profile": true,
    "chart": true,
    "monetary_section": true,
    "fiscal_section": true,
    "capital_markets_section": true,
    "employment_section": true,
    "statistical_table": true,
    "author": true,
    "publication_date": true
  },
  "all_expected_components_preserved": true,
  "table_heading_preserved": true,
  "table_row_checks": {
    "Real GDP": true,
    "Wholesale prices": true,
    "General government debt": true,
    "Current account balance": true,
    "External debt": true,
    "Gross reserves": t

In [32]:
# ============================================================
# 8. Save Branch C representation
# ============================================================
REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D5_branch_C_normalised_markdown.md"
)

REPRESENTATION_PATH.write_text(
    NORMALISED_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(
    REPRESENTATION_PATH
)

print(
    "Saved:",
    REPRESENTATION_PATH.name
)

print(
    "Representation SHA-256:",
    REPRESENTATION_SHA256
)


Saved: D5_branch_C_normalised_markdown.md
Representation SHA-256: 180241664a34d0e1155803ae20b0ec8d5a83a7a8c0fc309ef2d556eaa9353ac0


In [33]:
# ============================================================
# 9. Define fixed output schema
# ============================================================
EXPECTED_OUTPUT_STRUCTURE = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "records": [
        {
            "Category":
                None,

            "Indicator or Policy Area":
                None,

            "Value":
                None,

            "Unit":
                None,

            "Qualifier":
                None,

            "Reference Period":
                None,

            "Description":
                None,

            "Source Location":
                None
        }
    ]
}

print(
    json.dumps(
        EXPECTED_OUTPUT_STRUCTURE,
        indent=2,
        ensure_ascii=False
    )
)


{
  "document_id": "D5",
  "branch": "C",
  "records": [
    {
      "Category": null,
      "Indicator or Policy Area": null,
      "Value": null,
      "Unit": null,
      "Qualifier": null,
      "Reference Period": null,
      "Description": null,
      "Source Location": null
    }
  ]
}


In [34]:
# ============================================================
# 10. Define controlled Branch C extraction prompt
# ============================================================
#
# This mirrors the Branch B task and instruction strength.
# The expected 44-record count and expected category counts are
# NOT disclosed to the model.
# ============================================================

PROMPT_TEXT = f"""You are an information extraction assistant.

Extract the policy and quantitative records represented within the
defined scope of the attached deterministically normalised structural
Markdown document for the article “Booming India at risk of overheating.”

Treat the attached deterministically normalised structural Markdown
document as the only source of information.

Include:

1. Each principal policy measure introduced by the sentence
   “A combination of these four main policy measures is critical”.
2. Every item represented in the “India at a glance” country-profile
   region.
3. Every explicitly stated quantitative observation in the narrative
   article that belongs to the defined extraction scope.
4. The explicit quantitative statement represented in the “Taking off”
   chart caption.
5. Every numeric observation represented in the structurally
   reconstructed “Inflation risks” statistical table.

Exclude:

- values inferred or estimated from plotted chart lines;
- chart axis values;
- chart series dates and chart legend dates;
- page numbers and publication metadata;
- the photograph and photograph caption;
- promotional content;
- copyright text and publisher branding;
- values appearing only in source citations;
- qualitative statements that are not one of the principal policy
  measures;
- detailed policy sub-actions that merely elaborate a principal policy
  measure;
- Markdown representation metadata, source-block identifiers,
  coordinates and layout-region labels.

For every included record, extract:

- Category
- Indicator or Policy Area
- Value
- Unit
- Qualifier
- Reference Period
- Description
- Source Location

Category rules:

Use exactly one of:

- Main policy measure
- Country profile
- Narrative quantitative observation
- Statistical table observation

Indicator or Policy Area:

- Provide a concise source-grounded name for the represented policy
  area, profile item or quantitative indicator.
- Do not merge separate observations merely because they concern a
  similar topic.

Value:

- Preserve the explicitly represented source value.
- Use a JSON number for quantitative values.
- Use a JSON string for textual values.
- Keep negative numbers negative.
- Do not calculate, convert, reinterpret or estimate values.

Unit:

- Preserve the source-grounded measurement unit.
- Use "text" for textual policy or profile values.
- Do not silently correct source units.
- Preserve the unit printed for Gross reserves even if it appears
  unusual.

Qualifier:

- Preserve explicit approximation or inequality qualifiers such as
  "about", "around", "just over", "more than" and "nearly".
- Use null when no such qualifier is explicitly associated with the
  value.
- Do not use table-column status labels such as Prov. or Est. as
  Qualifier; those may be retained in Description where relevant.

Reference Period:

- Preserve explicitly associated fiscal years, calendar years,
  durations or relative periods.
- Use null when no explicit period is associated with the record.

Description:

- Provide a concise source-grounded explanation.
- Do not introduce external interpretation.

Source Location:

Use one of these exact labels when applicable:

- Page 1 — Four main policy measures
- Page 1 — India at a glance
- Page 1 — Opening narrative
- Page 1 — Taking off chart caption
- Page 1 — Ease up on the monetary accelerator
- Page 1 — Reduce debt to finance development
- Page 2 — Reduce debt to finance development
- Page 2 — Promote job growth and bolster the infrastructure
- Page 2 — Inflation risks table

Additional rules:

- Use the explicit structural cues in the Markdown while remaining
  grounded in source content.
- Distinguish narrative observations from statistical-table
  observations.
- Preserve decimal precision as represented.
- Preserve negative signs.
- Do not use external knowledge.
- Do not follow hyperlinks.
- Return one record for every included source observation.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names defined in the schema.

Expected JSON schema:
{json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
)}

Return only the JSON object.
""".strip()


PROMPT_PATH = (
    OUTPUT_DIR
    / "D5_branch_C_prompt.txt"
)

PROMPT_PATH.write_text(
    PROMPT_TEXT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)

print(
    PROMPT_TEXT
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)


You are an information extraction assistant.

Extract the policy and quantitative records represented within the
defined scope of the attached deterministically normalised structural
Markdown document for the article “Booming India at risk of overheating.”

Treat the attached deterministically normalised structural Markdown
document as the only source of information.

Include:

1. Each principal policy measure introduced by the sentence
   “A combination of these four main policy measures is critical”.
2. Every item represented in the “India at a glance” country-profile
   region.
3. Every explicitly stated quantitative observation in the narrative
   article that belongs to the defined extraction scope.
4. The explicit quantitative statement represented in the “Taking off”
   chart caption.
5. Every numeric observation represented in the structurally
   reconstructed “Inflation risks” statistical table.

Exclude:

- values inferred or estimated from plotted chart lines;
- chart axis v

In [35]:
# ============================================================
# 11. Create representation and experiment metadata
# ============================================================
REPRESENTATION_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "parent_B_representation_file":
        BRANCH_B_REPRESENTATION_PATH.name,

    "parent_B_representation_sha256":
        SOURCE_B_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "representation_type":
        "Complete Branch B structural Markdown with deterministic normalisation",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "complete_source_pdf_retained":
        True,

    "source_pages_retained":
        EXPECTED_PAGE_COUNT,

    "all_source_blocks_retained":
        True,

    "out_of_scope_content_retained":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "statistical_table_structurally_reconstructed":
        True,

    "normalisation_applied":
        True,

    "normalisation_operations": [
        "Unicode NFKC normalisation",
        "Unicode-space standardisation",
        "apostrophe standardisation",
        "dash/minus-glyph standardisation",
        "soft-hyphen removal",
        "line-ending standardisation",
        "horizontal whitespace normalisation"
    ],

    "paragraph_line_merging_applied":
        False,

    "line_break_hyphenation_repair_applied":
        False,

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "chart_values_estimated":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ]
}


REP_METADATA_PATH = (
    OUTPUT_DIR
    / "D5_branch_C_representation_metadata.json"
)

REP_METADATA_PATH.write_text(
    json.dumps(
        REPRESENTATION_METADATA,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "direct_document_ingestion":
        False,

    "structural_conversion_applied":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_content_retained":
        True,

    "out_of_scope_content_retained":
        True,

    "ocr_applied":
        False,

    "page_cropping_applied":
        False,

    "reference_values_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "manual_response_repair_permitted":
        False,

    "expected_output_format":
        "JSON object",

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "execution_environment":
        "Independent ChatGPT conversation",

    "model":
        "GPT-5.5",

    "created_at":
        datetime.now().isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "validation_status":
        "Pending Stage 4 Branch C validation against the fixed Stage 1 "
        "reference dataset using Branch A-frozen D5 comparison rules"
}


METADATA_PATH = (
    OUTPUT_DIR
    / "D5_branch_C_experiment_metadata.json"
)

METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        EXPERIMENT_METADATA,
        indent=2,
        ensure_ascii=False
    )
)


{
  "document_id": "D5",
  "document_name": "IMF Country Focus — Booming India at risk of overheating",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D5 - IMF “Country Focus” – India Article (1).pdf",
  "source_sha256": "ce59b2fb51cfcee5b579dd713c59ae5a5e14b57f66eab40d4cbaee151a62064d",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised structural Markdown",
  "representation_file": "D5_branch_C_normalised_markdown.md",
  "representation_sha256": "180241664a34d0e1155803ae20b0ec8d5a83a7a8c0fc309ef2d556eaa9353ac0",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "direct_document_ingestion": false,
  "structural_conversion_applied": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_source_content_retained": true,
  "out_of_scope_content_retained": true,
  "ocr_applied": false,
  "page_cropping_app

In [36]:
# ============================================================
# 12. Final pre-extraction control check
# ============================================================
PRECHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_identity_verified":
        SOURCE_HASH_MATCH,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "complete_source_content_retained":
        True,

    "page_sequence_preserved":
        page_sequence_preserved,

    "source_block_sequence_preserved":
        source_block_sequence_preserved,

    "statistical_table_preserved":
        all_table_rows_preserved
        and all_table_values_preserved,

    "representation_exists":
        REPRESENTATION_PATH.exists(),

    "prompt_exists":
        PROMPT_PATH.exists(),

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "ready_for_independent_llm_execution": bool(

        SOURCE_HASH_MATCH

        and PARENT_EQUIVALENCE_PASSED

        and normalisation_check[
            "normalisation_integrity_passed"
        ]

        and REPRESENTATION_PATH.exists()

        and PROMPT_PATH.exists()
    )
}


PRECHECK_PATH = (
    OUTPUT_DIR
    / "D5_branch_C_pre_extraction_check.json"
)

PRECHECK_PATH.write_text(
    json.dumps(
        PRECHECK,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        PRECHECK,
        indent=2,
        ensure_ascii=False
    )
)

if not PRECHECK[
    "ready_for_independent_llm_execution"
]:
    raise ValueError(
        "D5 Branch C is not ready for independent LLM execution."
    )


{
  "document_id": "D5",
  "branch": "C",
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "complete_source_content_retained": true,
  "page_sequence_preserved": true,
  "source_block_sequence_preserved": true,
  "statistical_table_preserved": true,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "expected_category_counts_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [37]:
# ============================================================
# 13. Download pre-extraction Branch C artefacts
# ============================================================
for path in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    REP_METADATA_PATH,
    METADATA_PATH,
    PRECHECK_PATH
]:
    files.download(
        path
    )

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D5_branch_C_normalised_markdown.md.\n"
    "3. Submit D5_branch_C_prompt.txt exactly once.\n"
    "4. Do not upload the original PDF, Branch B artefacts, Stage 1 "
    "reference values, or previous extraction outputs.\n"
    "5. Do not manually repair, correct, or regenerate the response.\n"
    "6. Save the complete response exactly as returned in a plain-text file."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Independent extraction instructions:
1. Open a new independent ChatGPT conversation.
2. Upload ONLY D5_branch_C_normalised_markdown.md.
3. Submit D5_branch_C_prompt.txt exactly once.
4. Do not upload the original PDF, Branch B artefacts, Stage 1 reference values, or previous extraction outputs.
5. Do not manually repair, correct, or regenerate the response.
6. Save the complete response exactly as returned in a plain-text file.


In [38]:
# ============================================================
# 14. Upload and preserve the complete raw Branch C response
# ============================================================
uploaded_response = files.upload()

if len(uploaded_response) != 1:
    raise ValueError(
        "Upload exactly one complete raw Branch C response text file."
    )

RAW_RESPONSE_SOURCE = Path(
    next(
        iter(
            uploaded_response
        )
    )
)

RAW_RESPONSE_TEXT = (
    RAW_RESPONSE_SOURCE.read_text(
        encoding="utf-8"
    )
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D5_branch_C_raw_response.txt"
)

RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)

print(
    "Raw response preserved unchanged."
)

print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)


Saving D5_branch_C_raw_response.txt to D5_branch_C_raw_response.txt
Raw response preserved unchanged.
Raw response SHA-256: 52e334bce9acb03b3994c7f4e62f5d60b130840b2893f99a3d3f4f6c28e02d8f


In [39]:
# ============================================================
# 15. Parse the raw response without repair
# ============================================================
valid_json = True
json_error = None
parsed_extraction = None

try:
    parsed_extraction = json.loads(
        RAW_RESPONSE_TEXT
    )

except json.JSONDecodeError as exc:
    valid_json = False
    json_error = str(exc)


print(
    "Valid JSON:",
    valid_json
)

if json_error:
    print(
        "JSON parsing error:",
        json_error
    )


Valid JSON: True


In [40]:
# ============================================================
# 16. Validate top-level output structure
# ============================================================
top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_extraction,
        dict
    )
)

document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_extraction
)

document_id_correct = (
    document_id_present
    and parsed_extraction.get(
        "document_id"
    )
    == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_extraction
)

branch_correct = (
    branch_present
    and parsed_extraction.get(
        "branch"
    )
    == BRANCH
)

records_present = (
    top_level_object_valid
    and "records"
    in parsed_extraction
)

records_is_list = (
    records_present
    and isinstance(
        parsed_extraction.get(
            "records"
        ),
        list
    )
)

records_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list
])

records = (
    parsed_extraction[
        "records"
    ]
    if records_evaluable
    else []
)

observed_record_count = (
    len(records)
    if records_evaluable
    else None
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed records:",
    observed_record_count
)


Records evaluable: True
Observed records: 46


In [41]:
# ============================================================
# 17. Validate record schemas and field types
# ============================================================
record_structure_issues = []
field_type_issues = []

for record_index, record in enumerate(
    records
):

    if not isinstance(
        record,
        dict
    ):

        record_structure_issues.append({
            "record_index":
                record_index,

            "issue":
                "Record is not a JSON object"
        })

        continue


    actual_fields = set(
        record.keys()
    )

    expected_fields = set(
        EXPECTED_FIELDS
    )

    missing_fields = sorted(
        expected_fields
        - actual_fields
    )

    additional_fields = sorted(
        actual_fields
        - expected_fields
    )

    if (
        missing_fields
        or additional_fields
    ):

        record_structure_issues.append({
            "record_index":
                record_index,

            "missing_fields":
                missing_fields,

            "additional_fields":
                additional_fields
        })


    for field in [
        "Category",
        "Indicator or Policy Area",
        "Unit",
        "Qualifier",
        "Reference Period",
        "Description",
        "Source Location"
    ]:

        value = record.get(
            field
        )

        if (
            value is not None
            and not isinstance(
                value,
                str
            )
        ):

            field_type_issues.append({
                "record_index":
                    record_index,

                "field":
                    field,

                "observed_type":
                    type(value).__name__
            })


    value = record.get(
        "Value"
    )

    if (
        isinstance(
            value,
            bool
        )

        or (
            value is not None
            and not isinstance(
                value,
                (
                    str,
                    int,
                    float
                )
            )
        )
    ):

        field_type_issues.append({
            "record_index":
                record_index,

            "field":
                "Value",

            "observed_type":
                type(value).__name__
        })


records_with_structure_issues = len(
    record_structure_issues
)

records_with_type_issues = len({
    item["record_index"]
    for item in field_type_issues
})

print(
    "Records with structure issues:",
    records_with_structure_issues
)

print(
    "Records with type issues:",
    records_with_type_issues
)


Records with structure issues: 0
Records with type issues: 0


In [42]:
# ============================================================
# 18. Scope/category/table diagnostics separate from schema validity
# ============================================================
record_count_valid = (
    observed_record_count
    == EXPECTED_RECORD_COUNT
    if records_evaluable
    else None
)

category_issues = []

observed_category_counts = (
    {
        category: sum(
            1
            for record in records
            if (
                isinstance(
                    record,
                    dict
                )
                and record.get(
                    "Category"
                )
                == category
            )
        )
        for category in EXPECTED_CATEGORY_COUNTS
    }
    if records_evaluable
    else None
)

if records_evaluable:

    for record_index, record in enumerate(
        records
    ):

        if not isinstance(
            record,
            dict
        ):
            continue

        if record.get(
            "Category"
        ) not in ALLOWED_CATEGORIES:

            category_issues.append({
                "record_index":
                    record_index,

                "observed_category":
                    record.get(
                        "Category"
                    )
            })


category_counts_valid = (
    observed_category_counts
    == EXPECTED_CATEGORY_COUNTS
    if records_evaluable
    else None
)


missing_values_by_field = (
    {
        field: sum(
            1
            for record in records
            if (
                not isinstance(
                    record,
                    dict
                )
                or field not in record
                or record.get(
                    field
                )
                is None
            )
        )
        for field in EXPECTED_FIELDS
    }
    if records_evaluable
    else None
)


def record_key(record):
    return (
        record.get(
            "Category"
        ),
        record.get(
            "Indicator or Policy Area"
        ),
        record.get(
            "Reference Period"
        ),
        record.get(
            "Source Location"
        )
    )


duplicate_record_keys = []

if records_evaluable:

    keys = [
        record_key(
            record
        )
        for record in records
        if isinstance(
            record,
            dict
        )
    ]

    duplicate_record_keys = sorted(
        {
            key
            for key in keys
            if keys.count(
                key
            )
            > 1
        },
        key=str
    )


table_records = [
    record
    for record in records
    if (
        isinstance(
            record,
            dict
        )
        and record.get(
            "Category"
        )
        == "Statistical table observation"
    )
]


table_record_count_valid = (
    len(
        table_records
    )
    == EXPECTED_CATEGORY_COUNTS[
        "Statistical table observation"
    ]
    if records_evaluable
    else None
)


scope_complete = bool(
    record_count_valid
    and category_counts_valid
    and table_record_count_valid
) if records_evaluable else False


CONTENT_DIAGNOSTICS = {
    "record_count_matches_reference":
        record_count_valid,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match_reference":
        category_counts_valid,

    "invalid_category_count":
        len(
            category_issues
        )
        if records_evaluable
        else None,

    "duplicate_record_key_count":
        len(
            duplicate_record_keys
        )
        if records_evaluable
        else None,

    "table_record_count":
        len(
            table_records
        )
        if records_evaluable
        else None,

    "table_record_count_valid":
        table_record_count_valid,

    "missing_values_by_field":
        missing_values_by_field
}


print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        indent=2,
        ensure_ascii=False
    )
)


{
  "record_count_matches_reference": false,
  "observed_category_counts": {
    "Main policy measure": 4,
    "Country profile": 6,
    "Narrative quantitative observation": 18,
    "Statistical table observation": 18
  },
  "category_counts_match_reference": false,
  "invalid_category_count": 0,
  "duplicate_record_key_count": 0,
  "table_record_count": 18,
  "table_record_count_valid": true,
  "missing_values_by_field": {
    "Category": 0,
    "Indicator or Policy Area": 0,
    "Value": 0,
    "Unit": 0,
    "Qualifier": 37,
    "Reference Period": 16,
    "Description": 0,
    "Source Location": 0
  }
}


In [43]:
# ============================================================
# 19. Keep schema validity separate from scope completeness
# ============================================================
schema_validity = bool(

    valid_json

    and top_level_object_valid

    and document_id_present

    and document_id_correct

    and branch_present

    and branch_correct

    and records_present

    and records_is_list

    and (
        records_with_structure_issues
        == 0
    )

    and (
        records_with_type_issues
        == 0
    )
)


STRUCTURE_CHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "valid_json":
        bool(
            valid_json
        ),

    "json_error":
        json_error,

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_present":
        bool(
            document_id_present
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_present":
        bool(
            branch_present
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_present":
        bool(
            records_present
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "schema_validity":
        bool(
            schema_validity
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_valid":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_valid":
        category_counts_valid,

    "scope_complete":
        bool(
            scope_complete
        ),

    "records_with_structure_issues":
        (
            records_with_structure_issues
            if records_evaluable
            else None
        ),

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "records_with_type_issues":
        (
            records_with_type_issues
            if records_evaluable
            else None
        ),

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "content_diagnostics":
        CONTENT_DIAGNOSTICS
}


STRUCTURE_CHECK_PATH = (
    OUTPUT_DIR
    / "D5_branch_C_structure_check.json"
)

STRUCTURE_CHECK_PATH.write_text(
    json.dumps(
        STRUCTURE_CHECK,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        STRUCTURE_CHECK,
        indent=2,
        ensure_ascii=False
    )
)


{
  "document_id": "D5",
  "branch": "C",
  "valid_json": true,
  "json_error": null,
  "top_level_object_valid": true,
  "document_id_present": true,
  "document_id_correct": true,
  "branch_present": true,
  "branch_correct": true,
  "records_present": true,
  "records_is_list": true,
  "records_evaluable": true,
  "schema_validity": true,
  "expected_record_count": 44,
  "observed_record_count": 46,
  "record_count_valid": false,
  "expected_category_counts": {
    "Main policy measure": 4,
    "Country profile": 6,
    "Narrative quantitative observation": 16,
    "Statistical table observation": 18
  },
  "observed_category_counts": {
    "Main policy measure": 4,
    "Country profile": 6,
    "Narrative quantitative observation": 18,
    "Statistical table observation": 18
  },
  "category_counts_valid": false,
  "scope_complete": false,
  "records_with_structure_issues": 0,
  "record_structure_issues": [],
  "records_with_type_issues": 0,
  "field_type_issues": [],
  "content_di

In [44]:
# ============================================================
# 20. Preserve parsed extraction only when JSON structure is evaluable
# ============================================================
PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D5_branch_C_parsed_extraction.json"
)

if records_evaluable:

    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            parsed_extraction,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    PARSED_EXTRACTION_SHA256 = (
        sha256_file(
            PARSED_EXTRACTION_PATH
        )
    )

    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:

    PARSED_EXTRACTION_SHA256 = None

    print(
        "No parsed extraction created because the preserved response "
        "does not contain an evaluable JSON records structure."
    )


Parsed extraction saved: D5_branch_C_parsed_extraction.json


In [45]:
# ============================================================
# 21. Create final Branch C experiment summary
# ============================================================
EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_content_retained":
        True,

    "out_of_scope_content_retained":
        True,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "raw_response_preserved":
        True,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "valid_json":
        bool(
            valid_json
        ),

    "schema_validity":
        bool(
            schema_validity
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "scope_complete":
        bool(
            scope_complete
        ),

    "records_with_structure_issues":
        (
            records_with_structure_issues
            if records_evaluable
            else None
        ),

    "records_with_type_issues":
        (
            records_with_type_issues
            if records_evaluable
            else None
        ),

    "parsed_extraction_created":
        bool(
            records_evaluable
        ),

    "parsed_extraction_sha256":
        PARSED_EXTRACTION_SHA256,

    "accuracy_validation_completed":
        False,

    "validation_status":
        (
            "Pending Stage 4 Branch C validation against the fixed "
            "Stage 1 reference dataset using Branch A-frozen D5 "
            "comparison rules"
            if records_evaluable
            else
            "Not content-evaluable because the preserved raw Branch C "
            "response does not contain an evaluable JSON records structure"
        )
}


SUMMARY_PATH = (
    OUTPUT_DIR
    / "D5_branch_C_experiment_summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        indent=2,
        ensure_ascii=False
    )
)


{
  "document_id": "D5",
  "document_name": "IMF Country Focus — Booming India at risk of overheating",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D5 - IMF “Country Focus” – India Article (1).pdf",
  "source_sha256": "ce59b2fb51cfcee5b579dd713c59ae5a5e14b57f66eab40d4cbaee151a62064d",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised structural Markdown",
  "representation_file": "D5_branch_C_normalised_markdown.md",
  "representation_sha256": "180241664a34d0e1155803ae20b0ec8d5a83a7a8c0fc309ef2d556eaa9353ac0",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_source_content_retained": true,
  "out_of_scope_content_retained": true,
  "reference_values_used_for_transformation": false,
  "expected_record_count_disclosed_to_model": false,
  "expected_categ

In [46]:
# ============================================================
# 22. Final artefact inventory and download
# ============================================================
artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    REP_METADATA_PATH,
    METADATA_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    SUMMARY_PATH
]

if records_evaluable:
    artefacts.append(
        PARSED_EXTRACTION_PATH
    )

print(
    "Final D5 Branch C artefacts:"
)

for path in artefacts:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )


for path in artefacts:

    if path.exists():
        files.download(
            path
        )


Final D5 Branch C artefacts:
- D5_branch_C_parent_B_equivalence_check.json | exists: True
- D5_branch_C_normalisation_check.json | exists: True
- D5_branch_C_normalised_markdown.md | exists: True
- D5_branch_C_prompt.txt | exists: True
- D5_branch_C_representation_metadata.json | exists: True
- D5_branch_C_experiment_metadata.json | exists: True
- D5_branch_C_pre_extraction_check.json | exists: True
- D5_branch_C_raw_response.txt | exists: True
- D5_branch_C_structure_check.json | exists: True
- D5_branch_C_experiment_summary.json | exists: True
- D5_branch_C_parsed_extraction.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>